# **Imports**

In [1]:
from dataclasses import dataclass
from typing import Literal
import dataclasses
import glob
import os
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, dataset

from torchvision import datasets
from torchvision.transforms import (
    Compose,
    Normalize,
    Resize,
    ToTensor,
    ToPILImage,
)

import wandb
from kaggle_secrets import UserSecretsClient

# **Initilize WandB**

In [2]:
user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ashiklibu1911 (ashiklibu1911-national-chung-cheng-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Current Pytorch Verison in kaggle envronment dose not suppor the P00 GPU. If you want to run the code on P100, you need to downgrade Pytorch verison and restart the kaggle kernel.

In [3]:
# !pip uninstall -y torch torchvision torchaudio numpy

# !pip install \
# torch==2.2.2 \
# torchvision==0.17.2 \
# torchaudio==2.2.2 \
# numpy==1.26.4

# **AlexNet Model (Not exactly the same as the original paper, but similar)**

In [4]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 96, kernel_size = 11, stride = 4),
            nn.ReLU(),
            nn.BatchNorm2d(96),
            nn.MaxPool2d(kernel_size = 3, stride = 2),
            nn.Conv2d(in_channels = 96, out_channels = 256, kernel_size = 5, padding = 2),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size = 3, stride = 2),
            nn.Conv2d(in_channels = 256, out_channels = 384, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels = 384, out_channels = 384, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels = 384, out_channels = 256, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2),
            )
        self.linear = nn.Sequential(
            nn.Linear(in_features = 6400, out_features = 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 10)
        )
                        
    def forward(self, x):
        x = self.conv(x).flatten(1)
        return self.linear(x)
        
    def num_parameters(self):
        count = 0
        for layer in self.parameters():
            count += layer.numel()
        return count

# if __name__ == "__main__":
#     data = torch.rand((10, 3, 224, 224))
#     model = AlexNet()
#     print(model(data).shape)
#     print(model.num_parameters())



# **Custom Dataset Class**

In [5]:
class Dataset:

    def __init__(self, config):
        self.config = config

        self._build_dataset()
        self._build_dataloader()

    def _build_transform(self):
        transforms = [
            Resize((self.config.img_size, self.config.img_size)),
            ToTensor(),
        ]

        # CIFAR-10 mean & std
        if self.config.normalize:
            transforms.append(
                Normalize(
                    mean=(0.4914, 0.4822, 0.4465),
                    std=(0.2470, 0.2435, 0.2616),
                )
            )

        return Compose(transforms)

    def _build_dataset(self):
        transform = self._build_transform()

        self.train_dataset = datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=transform,
        )

        self.test_dataset = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform=transform,
        )
        self.test_infer = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
        )
        self.classes = self.train_dataset.classes
        self.idToclasses = {v: k for k, v in self.train_dataset.class_to_idx.items()}

    def _build_dataloader(self):

        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.train_shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
        )

        self.test_loader = DataLoader(
            self.test_dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.test_shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
        )

    def __len__(self):
        return len(self.train_dataset)

    def num_classes(self):
        return len(self.classes)

# **Configurations**

In [6]:
@dataclass
class TrainConfig:
    # Training
    epochs: int = 100
    lr: float = 1e-3
    device: Literal["cuda", "cpu"] = "cuda"

    # Checkpoints
    load_from_checkpoint: bool = False
    checkpoint_path: str = "./checkpoints"

    save_best_model: bool = True
    save_last_model: bool = True

    # Monitor metric for best model
    monitor: Literal["val_loss", "val_acc"] = "val_acc"

    # Plots
    save_plots: bool = True
    plot_path: str = "./plots"

    # WandB
    wandb_monitor: bool = True
    project_name: str = "AlexNet"
    run_name: str = "alextnet-cifar-10"

    img_size: int = 224

    save_csv: bool = True
    csv_path: str = "log.csv"


@dataclass
class DatasetConfig:
    # Dataset
    img_size: int = 224
    batch_size: int = 64

    # DataLoader
    train_shuffle: bool = True
    test_shuffle: bool = False
    num_workers: int = 4
    pin_memory: bool = True

    # Transform
    normalize: bool = True


@dataclass
class InferenceConfig:
    img_size: int = 224
    load_from_checkpoint: bool = True
    checkpoint_path: str = "./checkpoints/best.pt"
    dataset_images_path: str = "./images"
    plot_file_name: str = "sample.png"
    
    

    

# **Custom Trainer Class**

In [7]:
class Trainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device(
            "cuda"
            if config.device == "cuda" and torch.cuda.is_available()
            else "cpu"
        )
        self.history = {
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": [],
        }

        if self.config.wandb_monitor:
            wandb.init(project = self.config.project_name,
                        name = self.config.run_name,
                        config = vars(config))

        if self.config.monitor == "val_acc":
            self.best_metric = -float("inf")
        else:
            self.best_metric = float("inf")

    def train(self, model, dataset):
        model = model.to(self.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(
            model.parameters(),
            lr=self.config.lr,
        )
        start_epoch = 0
        if self.config.load_from_checkpoint:
            start_epoch = self._load_checkpoint(
                model,
                optimizer,
            )

        for epoch in range(start_epoch, self.config.epochs):

            print(f"\nEpoch [{epoch+1}/{self.config.epochs}]")

            train_loss, train_acc = self._train_epoch(
                model,
                dataset.train_loader,
                criterion,
                optimizer,
            )

            val_loss, val_acc, _, _ = self._validate_epoch(
                model,
                dataset.test_loader,
                criterion,
            )

            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)

            if self.config.wandb_monitor:
                wandb.log({"epoch" : epoch + 1,
                        "train/loss" : train_loss,
                        "train/accuracy" : train_acc,
                        "test/loss" : val_loss,
                        "test/accuracy": val_acc})
            print(
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.2f}% | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.2f}%"
            )

            if self.config.save_last_model:
                self._save_checkpoint(
                    model,
                    optimizer,
                    epoch + 1,
                    "last.pt",
                )

            if self.config.save_best_model:

                metric = (
                    val_acc
                    if self.config.monitor == "val_acc"
                    else val_loss
                )

                improved = (
                    metric > self.best_metric
                    if self.config.monitor == "val_acc"
                    else metric < self.best_metric
                )

                if improved:
                    self.best_metric = metric

                    self._save_checkpoint(
                        model,
                        optimizer,
                        epoch + 1,
                        "best.pt",
                    )

                    if self.config.wandb_monitor:
                        wandb.log({
                            "best_val_accuracy": val_acc,
                            "best_val_loss" : val_loss
                        })

        _ = self._load_checkpoint(model, optimizer, best = True)

        if self.config.save_plots:
            self._plot_history()
            self._plot_confusion_matrix(model, dataset, criterion)
        
        if self.config.save_csv:
            self._save_csv()

        if self.config.wandb_monitor:
            self._upload_models_to_wandb()
            
        
        


        return self.history

    def _train_epoch(
        self,
        model,
        loader,
        criterion,
        optimizer,
    ):

        model.train()

        running_loss = 0
        correct = 0
        total = 0
        loop = tqdm(loader)
        for images, labels in loop:
            images = images.to(self.device)
            labels = labels.to(self.device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            loop.set_postfix(
                loss=running_loss / (loop.n + 1),
                acc=100 * correct / total,
            )
        epoch_loss = running_loss / len(loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc

    @torch.no_grad()
    def _validate_epoch(
        self,
        model,
        loader,
        criterion,
    ):
        model.eval()
        running_loss = 0
        correct = 0
        total = 0
        actual, predict =[], []
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            actual.append(predicted)
            predict.append(labels)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        epoch_loss = running_loss / len(loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc, actual, predict
    def _save_checkpoint(
        self,
        model,
        optimizer,
        epoch,
        filename,
    ):
        os.makedirs(
            self.config.checkpoint_path,
            exist_ok=True,
        )
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            os.path.join(
                self.config.checkpoint_path,
                filename,
            ),
        )

    def _load_checkpoint(
        self,
        model,
        optimizer,
        best = False
    ):

        checkpoint = torch.load(
            os.path.join(
                self.config.checkpoint_path,
                "best.pt" if best else "last.pt",
            ),
            map_location=self.device,
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )
        print("Checkpoint Loaded")
        return checkpoint["epoch"]

    def _upload_models_to_wandb(self):

        best_path = os.path.join(
            self.config.checkpoint_path,
            "best.pt"
        )

        last_path = os.path.join(
            self.config.checkpoint_path,
            "last.pt"
        )


        # Upload best model
        if os.path.exists(best_path):

            best_artifact = wandb.Artifact(
                name="best-model",
                type="model",
                description="Best validation performance model"
            )

            best_artifact.add_file(best_path)

            wandb.log_artifact(best_artifact)


        # Upload last model
        if os.path.exists(last_path):

            last_artifact = wandb.Artifact(
                name="last-model",
                type="model",
                description="Final epoch model"
            )

            last_artifact.add_file(last_path)

            wandb.log_artifact(last_artifact)
    
    def _plot_history(self):
        os.makedirs(
            self.config.plot_path,
            exist_ok=True,
        )
        epochs = range(
            1,
            len(self.history["train_loss"]) + 1,
        )
        plt.figure(figsize=(8, 5))
        plt.plot(
            epochs,
            self.history["train_loss"],
            label="Train",
        )
        plt.plot(
            epochs,
            self.history["val_loss"],
            label="Validation",
        )
        loss_path = os.path.join(self.config.plot_path, "loss.png")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Loss")
        plt.legend()
        plt.savefig(loss_path)

        plt.close()

        plt.figure(figsize=(8, 5))

        plt.plot(
            epochs,
            self.history["train_acc"],
            label="Train",
        )
        plt.plot(
            epochs,
            self.history["val_acc"],
            label="Validation",
        )
        acc_path = os.path.join(self.config.plot_path,"accuracy.png")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy (%)")
        plt.title("Accuracy")
        plt.legend()
        plt.savefig(acc_path)
        plt.close()

        if self.config.wandb_monitor:
            wandb.log({
                "loss_curve": wandb.Image(loss_path),
                "acc_curve" : wandb.Image(acc_path)
            })

    def _plot_confusion_matrix(self, model, dataset, criterion):
        idToClass = dataset.idToclasses
        _, _, actual, predict = self._validate_epoch(model, dataset.test_loader, criterion)

        actual = [i for batch in actual for i in batch]
        predict = [i for batch in predict for i in batch]
        cm = [[0 for _ in range(len(idToClass))] for _ in range(len(idToClass))]
        for i, j in zip(actual, predict):
            cm[i-1][j-1] += 1
        cm = torch.tensor(cm)
        class_names = [k for k in idToClass.keys()]

        plt.figure(figsize = (7, 6))

        sns.heatmap(
            cm,
            annot = True,
            fmt = "d",
            cmap = "Blues",
            xticklabels = class_names,
            yticklabels = class_names
        )


        confusion_mat_path = os.path.join(self.config.plot_path, "confusion_mat.png")
        plt.xlabel("predicted_labels")
        plt.ylabel("Actual labels")
        plt.title("Confusion Matrix")
        plt.tight_layout()
        plt.savefig(confusion_mat_path)
        plt.close()

        if self.config.wandb_monitor:
            wandb.log(
                {
                    "confussion_mat" : wandb.Image(confusion_mat_path)
                }
            )
    
    def _save_csv(self):
        df = pd.DataFrame([self.history])
        df.to_csv(self.config.csv_path)
        if self.config.wandb_monitor:
            csv_artifact = wandb.Artifact("Csv_log", type = "dataset", description = "log csv file")
            csv_artifact.add_file(self.config.csv_path)
            wandb.log_artifact(csv_artifact)

        

if __name__ == "__main__":

    dataset_config = DatasetConfig()
    train_config = TrainConfig()
    model = AlexNet()
    loader = Dataset(dataset_config)

    trainer = Trainer(train_config)

    trainer.train(model, loader)


100%|██████████| 170M/170M [28:57<00:00, 98.1kB/s]
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260812_155601-65kifevt
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run alextnet-cifar-10
wandb: ⭐️ View project at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/AlexNet
wandb: 🚀 View run at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/AlexNet/runs/65kifevt



Epoch [1/100]


100%|██████████| 782/782 [01:13<00:00, 10.62it/s, acc=35.5, loss=1.74]


Train Loss: 1.7449 | Train Acc: 35.51% | Val Loss: 1.4980 | Val Acc: 44.93%

Epoch [2/100]


100%|██████████| 782/782 [01:16<00:00, 10.27it/s, acc=48.7, loss=1.42]


...

Train Loss: 0.2566 | Train Acc: 92.88% | Val Loss: 1.0986 | Val Acc: 76.15%

Epoch [99/100]


100%|██████████| 782/782 [01:17<00:00, 10.10it/s, acc=94.6, loss=0.188]


Train Loss: 0.1875 | Train Acc: 94.64% | Val Loss: 1.0889 | Val Acc: 76.85%

Epoch [100/100]


100%|██████████| 782/782 [01:17<00:00, 10.10it/s, acc=94.6, loss=0.194]


Train Loss: 0.1938 | Train Acc: 94.58% | Val Loss: 1.2547 | Val Acc: 75.50%
Checkpoint Loaded


# **Inference(Automatically Pic images from test dataset)**

**Load Model**

In [8]:
model = AlexNet()
state = torch.load("/kaggle/working/checkpoints/best.pt")
model.load_state_dict(state["model_state_dict"])
loader = Dataset(dataset_config)

In [9]:
class classify:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.transforms = Compose([ToTensor(),
                            Resize((self.config.img_size, self.config.img_size)),
                            Normalize(mean=(0.4914, 0.4822, 0.4465),\
                                 std=(0.2470, 0.2435, 0.2616))])
        self.resize = Resize((self.config.img_size, self.config.img_size))

    def predict(self, idToClass: dict, images: list | str | None = None, file_name: str = "sample.png"):
        if images == None:
            images = self.pic_images_from_testset()
            images = glob.glob(self.config.dataset_images_path + "/*.jpg")
        if isinstance(images, list):
            fig, plots = plt.subplots(2, len(images)//2, figsize = (10, 5))
            plots = plots.flatten()
            if isinstance(images[0], str):
                for i in range(len(images)):
                    out = self._predict(images[i])
                    plots[i].imshow(Image.open(images[i]).convert("RGB"))
                    plots[i].set_title(f"Class : {idToClass[out]}")
                    plots[i].axis("off")
                plt.tight_layout()
                plt.savefig(file_name)
                plt.close()
        elif isinstance(images, str):
            out = self._predict(images)
            plt.imshow(Image.open(images).convert("RGB"))
            plt.axis("off")
            plt.title(f"Class : {idToClass[out]}")
            plt.savefig(file_name)
        else:
            print("Currently not supported")
        if os.path.isfile(file_name):
            wandb.log({
                "prediction" : wandb.Image(file_name)
            })
            wandb.finish()
            
    
    def _predict(self, img):
        img = Image.open(img).convert("RGB")
        img = self.transforms(img).unsqueeze(0)
        out = self.model(img).argmax()
        return int(out)

    def pic_images_from_testset(self, loader = loader, count = 10, clear = False):
        if clear and os.path.isdir(self.config.dataset_images_path):
            shutil.rmtree(self.config.dataset_images_path)
        test_set = loader.test_infer
        os.makedirs(self.config.dataset_images_path, exist_ok = True)
        total = len(test_set)
        for i in range(count):
            idx = random.randint(0, total-1)
            img , label = test_set[idx]
            img = self.resize(img)
            img.save(f"{self.config.dataset_images_path}/test_{i+1}.jpg")
        return
       

In [10]:
infer = classify(model, InferenceConfig())
idToClasses = loader.idToclasses
infer.predict(idToClass = idToClasses)


wandb: updating run metadata
wandb: uploading history steps 126-126, summary
wandb: 
wandb: Run history:
wandb: best_val_accuracy ▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇█████████
wandb:     best_val_loss █▆▅▃▃▃▃▂▂▂▂▁▁▁▂▁▁▁▂▁▂▂▃▅
wandb:             epoch ▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇███
wandb:     test/accuracy ▁▃▄▅▆▆▆▇▇▇▇▇▇█▆█▇▇█▇████▇██▇████████████
wandb:         test/loss █▃▃▃▂▂▁▁▁▁▂▁▁▂▃▂▂▄▁▃▂▂▃▂▂▃▃▄▃▄▄▂▄▄▄▄▆▅▄▆
wandb:    train/accuracy ▁▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇████████████████████████
wandb:        train/loss █▇▇▆▅▅▅▄▄▄▃▃▂▂▂▂▂▄▂▂▂▁▂▂▁▂▁▁▁▁▁▂▁▂▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 76.87
wandb:     best_val_loss 1.21866
wandb:             epoch 100
wandb:     test/accuracy 75.5
wandb:         test/loss 1.25471
wandb:    train/accuracy 94.576
wandb:        train/loss 0.19379
wandb: 
wandb: 🚀 View run alextnet-cifar-10 at: https://wandb.ai/ashiklibu1911-national-chung-cheng-university/AlexNet/runs/65kifevt
wandb: ⭐️ View project at: https://wandb.ai/ashiklibu1911-national-chung-cheng-univ